# Amazon ML Challenge - BiLSTM + Transformer Fusion Architecture

## Overview
This notebook implements a sophisticated multimodal architecture:
1. **Structured Feature Engineering**: Parse catalog_content into structured fields
2. **Text Encoder**: BERT token-level embeddings → BiLSTM aggregation
3. **Image Encoder**: EfficientNet-B3 embeddings
4. **Fusion Transformer**: Cross-modal attention across text, image, and structured features
5. **Optuna Optimization**: Hyperparameter tuning on subset of data
6. **Log-transformed target**: Predict log1p(price), evaluate SMAPE on original scale

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
import re
import os
from PIL import Image
import timm
from transformers import AutoTokenizer, AutoModel
import optuna
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Configuration and Data Paths

In [ ]:
# Configuration
BASE_DIR = '/Users/gabi/Desktop/DL stuff/AMAZONML/student_resource'
TRAIN_CSV = f'{BASE_DIR}/dataset/train.csv'
TEST_CSV = f'{BASE_DIR}/dataset/test.csv'
TRAIN_IMAGE_DIR = f'{BASE_DIR}/dataset/train_images'  # Adjust to your actual path
TEST_IMAGE_DIR = f'{BASE_DIR}/dataset/test_images'    # Adjust to your actual path

# Model configuration
BERT_MODEL = 'bert-base-uncased'  # or 'sentence-transformers/all-mpnet-base-v2'
IMAGE_MODEL = 'efficientnet_b3'
MAX_TEXT_LENGTH = 256
IMAGE_SIZE = 224

# Training configuration
OPTUNA_TRIALS = 20
OPTUNA_DATA_FRACTION = 0.3  # Use 30% of training data for Optuna
BATCH_SIZE = 32
NUM_EPOCHS = 100
PATIENCE = 15

print("Configuration loaded successfully.")

## 2. Load Data

In [ ]:
# Load datasets
print("Loading datasets...")
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"\nTrain columns: {train_df.columns.tolist()}")
print(f"\nFirst row:")
print(train_df.head(1))
print(f"\nSample catalog_content:")
print(train_df['catalog_content'].iloc[0][:500])

## 3. Parse catalog_content into Structured Features

In [ ]:
# Unit normalization mappings
UNIT_MAPPINGS = {
    "ounces": "oz", "ounce": "oz", "oz": "oz",
    "pounds": "lb", "pound": "lb", "lbs": "lb", "lb": "lb",
    "grams": "g", "gram": "g", "gr": "g", "g": "g",
    "kilograms": "kg", "kilogram": "kg", "kg": "kg",
    "liters": "l", "liter": "l", "litre": "l", "ltr": "l", "l": "l",
    "milliliters": "ml", "milliliter": "ml", "millilitre": "ml", "ml": "ml",
    "fluid": "fl", "fl": "fl",
    "count": "count", "piece": "piece", "pieces": "piece",
    "pack": "pack", "packs": "pack",
    "bottle": "bottle", "bottles": "bottle",
    "box": "box", "boxes": "box",
    "jar": "jar", "jars": "jar",
    "tin": "tin", "tins": "tin",
    "bag": "bag", "bags": "bag",
    "packet": "packet", "packets": "packet",
    "sachet": "sachet", "sachets": "sachet"
}

def normalize_unit(unit_str):
    """Normalize unit string to canonical form"""
    if pd.isna(unit_str) or not unit_str:
        return 'count'
    unit_lower = str(unit_str).lower().strip()
    return UNIT_MAPPINGS.get(unit_lower, unit_lower)

def parse_catalog_content(text):
    """Parse catalog_content into structured fields"""
    if pd.isna(text):
        return {
            'item_name': '',
            'bullet_points': '',
            'product_description': '',
            'value': np.nan,
            'unit': 'count',
            'brand': '',
            'bullet_point_count': 0,
            'text_length': 0,
            'has_description': 0,
            'is_combo': 0
        }
    
    text = str(text)
    result = {}
    
    # Extract Item Name (first line or before first pattern)
    item_name_match = re.search(r'^(.+?)(?:\n|Item Name:|$)', text, re.MULTILINE)
    result['item_name'] = item_name_match.group(1).strip() if item_name_match else ''
    
    # Extract Bullet Points
    bullet_points = re.findall(r'Bullet Point \d+:\s*(.+?)(?=\n|Bullet Point|Product Description|$)', text, re.IGNORECASE)
    result['bullet_points'] = ' || '.join(bullet_points) if bullet_points else ''
    result['bullet_point_count'] = len(bullet_points)
    
    # Extract Product Description
    desc_match = re.search(r'Product Description:\s*(.+?)(?=\n[A-Z]|$)', text, re.IGNORECASE | re.DOTALL)
    result['product_description'] = desc_match.group(1).strip() if desc_match else ''
    result['has_description'] = 1 if desc_match else 0
    
    # Extract Value and Unit
    value_match = re.search(r'Value:\s*([\d,\.]+)\s*(\w+)?', text, re.IGNORECASE)
    if value_match:
        value_str = value_match.group(1).replace(',', '')
        result['value'] = float(value_str) if value_str else np.nan
        result['unit'] = normalize_unit(value_match.group(2)) if value_match.group(2) else 'count'
    else:
        # Try to find value with unit pattern
        fallback_match = re.search(r'([\d,\.]+)\s*(kg|g|lb|oz|ml|l|count|piece|pack|bottle)\b', text, re.IGNORECASE)
        if fallback_match:
            result['value'] = float(fallback_match.group(1).replace(',', ''))
            result['unit'] = normalize_unit(fallback_match.group(2))
        else:
            result['value'] = np.nan
            result['unit'] = 'count'
    
    # Check for combo/pack keywords
    combo_keywords = r'\b(pack|combo|set of|bundle|value pack|2-in-1|3 pack|multi pack|pack of|qty|quantity)\b'
    full_text = result['item_name'] + ' ' + result['bullet_points']
    result['is_combo'] = 1 if re.search(combo_keywords, full_text, re.IGNORECASE) else 0
    
    # Extract brand (first word of item name as fallback)
    result['brand'] = result['item_name'].split()[0] if result['item_name'] else 'unknown'
    
    # Text statistics
    result['text_length'] = len(text)
    
    return result

print("Parsing functions defined.")

In [ ]:
# Apply parsing to both train and test
print("Parsing catalog_content for train set...")
train_parsed = train_df['catalog_content'].apply(parse_catalog_content)
train_parsed_df = pd.DataFrame(train_parsed.tolist())

print("Parsing catalog_content for test set...")
test_parsed = test_df['catalog_content'].apply(parse_catalog_content)
test_parsed_df = pd.DataFrame(test_parsed.tolist())

# Merge with original dataframes
train_df = pd.concat([train_df, train_parsed_df], axis=1)
test_df = pd.concat([test_df, test_parsed_df], axis=1)

print(f"\nParsed features sample:")
print(train_parsed_df.head())
print(f"\nValue statistics:")
print(train_parsed_df['value'].describe())

## 4. Feature Engineering - Derived Numeric Columns

In [ ]:
def engineer_features(df):
    """Create derived numeric features"""
    df = df.copy()
    
    # Value in base units (grams for weight, ml for volume)
    df['value_numeric'] = df['value'].fillna(0)
    df['value_in_base_unit'] = df.apply(lambda row: convert_to_base_unit(row['value_numeric'], row['unit']), axis=1)
    
    # Extract quantity from item name
    df['quantity'] = df['item_name'].apply(extract_quantity)
    
    # Weight category
    df['weight_category'] = pd.cut(df['value_in_base_unit'], 
                                     bins=[-np.inf, 50, 200, 1000, np.inf],
                                     labels=['<50', '50-200', '200-1000', '>1000'])
    df['weight_category'] = df['weight_category'].astype(str)
    
    # Missing value flag
    df['numeric_value_missing'] = df['value'].isna().astype(int)
    
    # Log text length
    df['log_text_length'] = np.log1p(df['text_length'])
    
    return df

def convert_to_base_unit(value, unit):
    """Convert value to base unit (g for weight, ml for volume)"""
    if pd.isna(value) or value == 0:
        return 0
    
    conversion = {
        'kg': 1000, 'g': 1, 'mg': 0.001,
        'lb': 453.592, 'oz': 28.3495,
        'l': 1000, 'ml': 1,
        'count': value, 'piece': value, 'pack': value,
        'bottle': value, 'box': value, 'jar': value,
        'tin': value, 'bag': value, 'packet': value, 'sachet': value
    }
    
    return value * conversion.get(unit, 1)

def extract_quantity(item_name):
    """Extract quantity from item name (e.g., '2 pk', '12x', '6 pack')"""
    if pd.isna(item_name):
        return 1
    
    # Look for patterns like "2 pk", "12x", "6 pack"
    patterns = [
        r'(\d+)\s*(?:pk|pack|x|count|pcs|pieces)',
        r'(\d+)-pack',
        r'pack\s+of\s+(\d+)'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, str(item_name), re.IGNORECASE)
        if match:
            return int(match.group(1))
    
    return 1

# Apply feature engineering
print("Engineering features for train set...")
train_df = engineer_features(train_df)

print("Engineering features for test set...")
test_df = engineer_features(test_df)

print(f"\nEngineered features:")
print(train_df[['value_numeric', 'value_in_base_unit', 'quantity', 'weight_category']].head())
print(f"\nWeight category distribution:")
print(train_df['weight_category'].value_counts())

## 5. Encode Categorical Features

In [ ]:
# Encode categorical variables
categorical_features = ['unit', 'brand', 'weight_category']
label_encoders = {}

for feat in categorical_features:
    le = LabelEncoder()
    # Fit on combined train+test to ensure same encoding
    combined_values = pd.concat([train_df[feat], test_df[feat]]).astype(str)
    le.fit(combined_values)
    
    train_df[f'{feat}_encoded'] = le.transform(train_df[feat].astype(str))
    test_df[f'{feat}_encoded'] = le.transform(test_df[feat].astype(str))
    label_encoders[feat] = le
    
    print(f"{feat}: {len(le.classes_)} unique values")

print("\nCategorical encoding completed.")

## 6. Prepare Target Variable (Log Transform)

In [ ]:
# Apply log1p transform to target
train_df['price_log'] = np.log1p(train_df['price'])

print("Target variable statistics:")
print(f"Original price - Min: {train_df['price'].min():.2f}, Max: {train_df['price'].max():.2f}, Mean: {train_df['price'].mean():.2f}")
print(f"Log-transformed - Min: {train_df['price_log'].min():.2f}, Max: {train_df['price_log'].max():.2f}, Mean: {train_df['price_log'].mean():.2f}")

## 7. Train-Validation Split (80-20)

In [ ]:
# 80-20 split
train_data, val_data = train_test_split(train_df, test_size=0.2, random_state=42)

print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")

# For Optuna, create a smaller subset
optuna_data = train_data.sample(frac=OPTUNA_DATA_FRACTION, random_state=42)
optuna_train, optuna_val = train_test_split(optuna_data, test_size=0.2, random_state=42)

print(f"\nOptuna training set size: {len(optuna_train)}")
print(f"Optuna validation set size: {len(optuna_val)}")

## 8. Prepare Text Data for BERT

In [ ]:
# Initialize BERT tokenizer
print(f"Loading BERT tokenizer: {BERT_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)

def prepare_text(item_name, bullet_points, description):
    """Combine text fields for BERT input"""
    parts = []
    if item_name:
        parts.append(f"Item: {item_name}")
    if bullet_points:
        parts.append(f"Features: {bullet_points}")
    if description:
        parts.append(f"Description: {description}")
    return " ".join(parts) if parts else "No description available"

# Prepare text for all datasets
train_data['full_text'] = train_data.apply(
    lambda row: prepare_text(row['item_name'], row['bullet_points'], row['product_description']), axis=1
)
val_data['full_text'] = val_data.apply(
    lambda row: prepare_text(row['item_name'], row['bullet_points'], row['product_description']), axis=1
)
test_df['full_text'] = test_df.apply(
    lambda row: prepare_text(row['item_name'], row['bullet_points'], row['product_description']), axis=1
)

print(f"Sample full_text:")
print(train_data['full_text'].iloc[0][:300])

## 9. Define PyTorch Dataset and DataLoader

In [ ]:
# Define structured feature columns
STRUCTURED_FEATURES = [
    'value_numeric', 'value_in_base_unit', 'quantity',
    'bullet_point_count', 'log_text_length', 'has_description',
    'is_combo', 'numeric_value_missing',
    'unit_encoded', 'brand_encoded', 'weight_category_encoded'
]

# Note: For images, adjust based on your actual image availability
# If images are not available, we'll use dummy embeddings or skip image encoder

class MultimodalDataset(Dataset):
    """Dataset for multimodal price prediction"""
    def __init__(self, df, tokenizer, max_length=256, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_test = is_test
        
        # Prepare structured features
        self.structured_features = df[STRUCTURED_FEATURES].fillna(0).values.astype(np.float32)
        
        # Standardize structured features
        if not is_test:
            self.scaler = StandardScaler()
            self.structured_features = self.scaler.fit_transform(self.structured_features)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Tokenize text
        encoding = self.tokenizer(
            row['full_text'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Structured features
        struct_feats = torch.FloatTensor(self.structured_features[idx])
        
        # Image placeholder (replace with actual image loading if available)
        # For now, use a dummy tensor
        image_available = False  # Set to True if you have images
        if image_available and pd.notna(row.get('image_link')):
            # Load and preprocess image here
            image = torch.randn(3, IMAGE_SIZE, IMAGE_SIZE)  # Placeholder
        else:
            image = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)  # Dummy
        
        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'structured_features': struct_feats,
            'image': image
        }
        
        if not self.is_test:
            item['target'] = torch.FloatTensor([row['price_log']])
        
        return item

print("Dataset class defined.")

## 10. Define BiLSTM + Transformer Fusion Model

In [ ]:
class MultimodalPricePredictor(nn.Module):
    """
    BiLSTM + Transformer Fusion Model
    - BERT for token-level text embeddings
    - BiLSTM for sequential aggregation
    - EfficientNet for image embeddings (or dummy if unavailable)
    - Transformer fusion layer for cross-modal attention
    - MLP head for price prediction
    """
    def __init__(self, bert_model_name, n_structured_features, 
                 d_model=768, lstm_hidden=384, n_fusion_layers=2, 
                 n_heads=8, dropout=0.1, use_images=False):
        super(MultimodalPricePredictor, self).__init__()
        
        self.d_model = d_model
        self.use_images = use_images
        
        # 1. Text encoder (BERT)
        self.bert = AutoModel.from_pretrained(bert_model_name)
        # Freeze BERT layers (optionally unfreeze last few layers for fine-tuning)
        for param in self.bert.parameters():
            param.requires_grad = False
        # Unfreeze last 2 layers
        for param in self.bert.encoder.layer[-2:].parameters():
            param.requires_grad = True
        
        # 2. BiLSTM for text aggregation
        self.text_bilstm = nn.LSTM(
            input_size=d_model,
            hidden_size=lstm_hidden,
            num_layers=1,
            bidirectional=True,
            batch_first=True,
            dropout=0
        )
        self.text_projection = nn.Linear(lstm_hidden * 2, d_model)
        self.text_norm = nn.LayerNorm(d_model)
        
        # 3. Image encoder (EfficientNet)
        if self.use_images:
            self.image_encoder = timm.create_model(IMAGE_MODEL, pretrained=True, num_classes=0)
            image_feat_dim = self.image_encoder.num_features
            self.image_projection = nn.Linear(image_feat_dim, d_model)
            self.image_norm = nn.LayerNorm(d_model)
        
        # 4. Structured features branch
        self.struct_mlp = nn.Sequential(
            nn.Linear(n_structured_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, d_model)
        )
        self.struct_norm = nn.LayerNorm(d_model)
        
        # 5. Fusion Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        self.fusion_transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_fusion_layers)
        
        # 6. Output head
        self.output_mlp = nn.Sequential(
            nn.Linear(d_model, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )
        
    def forward(self, input_ids, attention_mask, structured_features, image=None):
        batch_size = input_ids.size(0)
        
        # 1. Text encoding with BERT
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = bert_output.last_hidden_state  # [batch, seq_len, 768]
        
        # 2. BiLSTM aggregation
        lstm_output, (h_n, c_n) = self.text_bilstm(token_embeddings)
        # Use final hidden state (concat forward + backward)
        text_repr = torch.cat([h_n[-2], h_n[-1]], dim=1)  # [batch, lstm_hidden*2]
        text_token = self.text_projection(text_repr)  # [batch, d_model]
        text_token = self.text_norm(text_token)
        text_token = text_token.unsqueeze(1)  # [batch, 1, d_model]
        
        # 3. Structured features
        struct_token = self.struct_mlp(structured_features)  # [batch, d_model]
        struct_token = self.struct_norm(struct_token)
        struct_token = struct_token.unsqueeze(1)  # [batch, 1, d_model]
        
        # 4. Image encoding (if available)
        if self.use_images and image is not None:
            image_features = self.image_encoder(image)  # [batch, image_feat_dim]
            image_token = self.image_projection(image_features)
            image_token = self.image_norm(image_token)
            image_token = image_token.unsqueeze(1)  # [batch, 1, d_model]
            
            # Combine tokens: [text, image, structured]
            fusion_input = torch.cat([text_token, image_token, struct_token], dim=1)  # [batch, 3, d_model]
        else:
            # Combine tokens: [text, structured]
            fusion_input = torch.cat([text_token, struct_token], dim=1)  # [batch, 2, d_model]
        
        # 5. Fusion Transformer
        fusion_output = self.fusion_transformer(fusion_input)  # [batch, num_tokens, d_model]
        
        # 6. Pool and predict
        pooled = fusion_output.mean(dim=1)  # [batch, d_model]
        output = self.output_mlp(pooled)  # [batch, 1]
        
        return output.squeeze(-1)

print("Model architecture defined.")

## 11. Define Evaluation Metrics (SMAPE Focus)

In [ ]:
def calculate_smape(y_true, y_pred):
    """Calculate Symmetric Mean Absolute Percentage Error"""
    denominator = (np.abs(y_true) + np.abs(y_pred))
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0
    return 100 * np.mean(diff)

def calculate_mape(y_true, y_pred):
    """Calculate Mean Absolute Percentage Error"""
    mask = y_true != 0
    return 100 * np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

def evaluate_model(y_true, y_pred, dataset_name="Validation"):
    """Calculate and display all evaluation metrics"""
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    smape = calculate_smape(y_true, y_pred)
    mape = calculate_mape(y_true, y_pred)
    
    print(f"\n{dataset_name} Set Metrics:")
    print(f"{'='*50}")
    print(f"MSE:    {mse:.4f}")
    print(f"MAE:    {mae:.4f}")
    print(f"SMAPE:  {smape:.4f}% ⭐ (Primary Metric)")
    print(f"MAPE:   {mape:.4f}%")
    print(f"R²:     {r2:.4f}")
    print(f"{'='*50}")
    
    return {
        'MSE': mse,
        'MAE': mae,
        'SMAPE': smape,
        'MAPE': mape,
        'R2': r2
    }

print("Evaluation metrics defined.")

## 12. Hyperparameter Optimization with Optuna

In [ ]:
# Optuna objective function
def objective(trial):
    """Optimize hyperparameters to minimize validation SMAPE"""
    
    # Suggest hyperparameters
    lstm_hidden = trial.suggest_int('lstm_hidden', 256, 512, step=128)
    n_fusion_layers = trial.suggest_int('n_fusion_layers', 2, 4)
    n_heads = trial.suggest_categorical('n_heads', [4, 8])
    dropout = trial.suggest_float('dropout', 0.1, 0.4)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    
    # Create datasets
    train_dataset = MultimodalDataset(optuna_train, tokenizer, MAX_TEXT_LENGTH)
    val_dataset = MultimodalDataset(optuna_val, tokenizer, MAX_TEXT_LENGTH)
    val_dataset.scaler = train_dataset.scaler  # Use same scaler
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    # Initialize model
    model = MultimodalPricePredictor(
        bert_model_name=BERT_MODEL,
        n_structured_features=len(STRUCTURED_FEATURES),
        d_model=768,
        lstm_hidden=lstm_hidden,
        n_fusion_layers=n_fusion_layers,
        n_heads=n_heads,
        dropout=dropout,
        use_images=False
    ).to(device)
    
    # Optimizer and loss
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    
    # Training loop (10 epochs for Optuna)
    best_smape = float('inf')
    patience = 3
    patience_counter = 0
    
    for epoch in range(10):
        model.train()
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            structured = batch['structured_features'].to(device)
            target = batch['target'].squeeze().to(device)
            
            optimizer.zero_grad()
            output = model(input_ids, attention_mask, structured)
            loss = criterion(output, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        
        # Validation
        model.eval()
        predictions_log = []
        targets_log = []
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                structured = batch['structured_features'].to(device)
                target = batch['target'].squeeze()
                
                output = model(input_ids, attention_mask, structured)
                predictions_log.extend(output.cpu().numpy())
                targets_log.extend(target.numpy())
        
        # Convert to original scale
        predictions_log = np.array(predictions_log)
        targets_log = np.array(targets_log)
        predictions = np.expm1(predictions_log)
        targets = np.expm1(targets_log)
        
        # Calculate SMAPE
        current_smape = calculate_smape(targets, predictions)
        
        # Report and prune
        trial.report(current_smape, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        # Early stopping
        if current_smape < best_smape:
            best_smape = current_smape
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    
    return best_smape

print("Optuna objective function defined.")

In [ ]:
# Run Optuna study
print(f"Starting Optuna hyperparameter optimization with {OPTUNA_TRIALS} trials...")
print(f"Using {len(optuna_train)} training samples and {len(optuna_val)} validation samples\n")

study = optuna.create_study(
    direction='minimize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3),
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

print("\n" + "="*70)
print("OPTUNA OPTIMIZATION COMPLETED!")
print("="*70)

best_trial = study.best_trial
print(f"\n🏆 Best Trial: {best_trial.number}")
print(f"📊 Best SMAPE: {best_trial.value:.4f}%")
print(f"\n🔧 Best Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  - {key}: {value}")

best_params = best_trial.params

## 13. Train Final Model with Optimized Hyperparameters

In [ ]:
# Create final datasets with full training data
train_dataset_final = MultimodalDataset(train_data, tokenizer, MAX_TEXT_LENGTH)
val_dataset_final = MultimodalDataset(val_data, tokenizer, MAX_TEXT_LENGTH)
val_dataset_final.scaler = train_dataset_final.scaler

train_loader_final = DataLoader(train_dataset_final, batch_size=best_params['batch_size'], shuffle=True, num_workers=0)
val_loader_final = DataLoader(val_dataset_final, batch_size=best_params['batch_size'], shuffle=False, num_workers=0)

# Initialize final model
final_model = MultimodalPricePredictor(
    bert_model_name=BERT_MODEL,
    n_structured_features=len(STRUCTURED_FEATURES),
    d_model=768,
    lstm_hidden=best_params['lstm_hidden'],
    n_fusion_layers=best_params['n_fusion_layers'],
    n_heads=best_params['n_heads'],
    dropout=best_params['dropout'],
    use_images=False
).to(device)

print(f"\nFinal model initialized with optimized hyperparameters")
print(f"Total parameters: {sum(p.numel() for p in final_model.parameters()):,}")

# Optimizer and loss
criterion = nn.MSELoss()
optimizer = optim.AdamW(final_model.parameters(), lr=best_params['learning_rate'], weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

# Training loop
best_val_loss = float('inf')
patience_counter = 0
train_losses = []
val_losses = []

print(f"\nStarting final training for up to {NUM_EPOCHS} epochs...")
print(f"Device: {device}\n")

for epoch in range(NUM_EPOCHS):
    # Training
    final_model.train()
    train_loss = 0.0
    
    for batch in train_loader_final:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        structured = batch['structured_features'].to(device)
        target = batch['target'].squeeze().to(device)
        
        optimizer.zero_grad()
        output = final_model(input_ids, attention_mask, structured)
        loss = criterion(output, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(final_model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader_final)
    train_losses.append(avg_train_loss)
    
    # Validation
    final_model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for batch in val_loader_final:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            structured = batch['structured_features'].to(device)
            target = batch['target'].squeeze().to(device)
            
            output = final_model(input_ids, attention_mask, structured)
            loss = criterion(output, target)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader_final)
    val_losses.append(avg_val_loss)
    
    scheduler.step(avg_val_loss)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
    
    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(final_model.state_dict(), 'best_multimodal_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print(f"\nTraining completed! Best val loss: {best_val_loss:.4f}")

## 14. Evaluate on Validation Set

In [ ]:
# Load best model
final_model.load_state_dict(torch.load('best_multimodal_model.pth'))
final_model.eval()

# Make predictions
val_predictions_log = []
val_targets_log = []

with torch.no_grad():
    for batch in val_loader_final:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        structured = batch['structured_features'].to(device)
        target = batch['target'].squeeze()
        
        output = final_model(input_ids, attention_mask, structured)
        val_predictions_log.extend(output.cpu().numpy())
        val_targets_log.extend(target.numpy())

val_predictions_log = np.array(val_predictions_log)
val_targets_log = np.array(val_targets_log)

# Inverse transform to original scale
val_predictions = np.expm1(val_predictions_log)
val_targets = np.expm1(val_targets_log)

# Evaluate
val_metrics = evaluate_model(val_targets, val_predictions, "Validation")

# Sample predictions
print("\n\nSample Predictions vs Actual:")
print(f"{'Actual':<12} {'Predicted':<12} {'Diff':<12} {'Error %':<10}")
print("-" * 50)
for i in range(min(15, len(val_targets))):
    actual = val_targets[i]
    pred = val_predictions[i]
    diff = pred - actual
    pct = 100 * abs(diff) / actual if actual != 0 else 0
    print(f"{actual:<12.2f} {pred:<12.2f} {diff:<12.2f} {pct:<10.2f}%")

## 15. Generate Test Set Predictions

In [ ]:
# Create test dataset
test_dataset = MultimodalDataset(test_df, tokenizer, MAX_TEXT_LENGTH, is_test=True)
test_dataset.scaler = train_dataset_final.scaler  # Use same scaler
test_loader = DataLoader(test_dataset, batch_size=best_params['batch_size'], shuffle=False, num_workers=0)

# Generate predictions
print("Generating test predictions...")
test_predictions_log = []

final_model.eval()
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        structured = batch['structured_features'].to(device)
        
        output = final_model(input_ids, attention_mask, structured)
        test_predictions_log.extend(output.cpu().numpy())

test_predictions_log = np.array(test_predictions_log)

# Inverse transform
test_predictions = np.expm1(test_predictions_log)

print(f"\nTest predictions generated: {len(test_predictions)}")
print(f"Statistics - Min: {test_predictions.min():.2f}, Max: {test_predictions.max():.2f}, Mean: {test_predictions.mean():.2f}")

## 16. Create Submission File

In [ ]:
# Create submission
submission_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': test_predictions
})

submission_file = 'test_predictions_multimodal.csv'
submission_df.to_csv(submission_file, index=False)

print(f"Submission file saved: {submission_file}")
print(f"\nSubmission preview:")
print(submission_df.head(10))
print(f"\nTotal predictions: {len(submission_df)}")

## 17. Final Summary

In [ ]:
print("=" * 80)
print("BILSTM + TRANSFORMER FUSION MODEL - FINAL SUMMARY")
print("=" * 80)

print("\n📊 MODEL ARCHITECTURE:")
print("  ├─ Text Encoder: BERT (bert-base-uncased)")
print("  │  └─ Token-level embeddings (768-dim)")
print("  ├─ Text Aggregator: BiLSTM")
print(f"  │  └─ Hidden size: {best_params['lstm_hidden']} (bidirectional)")
print("  ├─ Image Encoder: EfficientNet-B3 (optional, dummy used)")
print("  ├─ Structured Features: 11 engineered features")
print("  │  └─ MLP projection to 768-dim")
print("  ├─ Fusion: Transformer Encoder")
print(f"  │  ├─ Layers: {best_params['n_fusion_layers']}")
print(f"  │  ├─ Heads: {best_params['n_heads']}")
print(f"  │  └─ Dropout: {best_params['dropout']}")
print("  └─ Output: MLP Head (768 → 512 → 128 → 1)")

print("\n🔧 STRUCTURED FEATURES ENGINEERED:")
print("  • Parsed from catalog_content:")
print("    - Item Name, Bullet Points, Product Description")
print("    - Value (numeric), Unit (normalized), Brand")
print("  • Derived features:")
print("    - value_in_base_unit, quantity, weight_category")
print("    - bullet_point_count, text_length, has_description")
print("    - is_combo, numeric_value_missing")

print("\n⚙️ OPTIMIZED HYPERPARAMETERS (via Optuna):")
for key, value in best_params.items():
    print(f"  • {key}: {value}")

print("\n📈 DATASET:")
print(f"  • Total training samples: {len(train_df)}")
print(f"  • Training set: {len(train_data)} (80%)")
print(f"  • Validation set: {len(val_data)} (20%)")
print(f"  • Test set: {len(test_df)}")
print(f"  • Optuna subset: {len(optuna_data)} ({OPTUNA_DATA_FRACTION*100:.0f}% of train)")

print("\n🎯 VALIDATION PERFORMANCE:")
print(f"  • MSE:    {val_metrics['MSE']:.4f}")
print(f"  • MAE:    {val_metrics['MAE']:.4f}")
print(f"  • SMAPE:  {val_metrics['SMAPE']:.4f}% ⭐ (Primary Metric)")
print(f"  • MAPE:   {val_metrics['MAPE']:.4f}%")
print(f"  • R²:     {val_metrics['R2']:.4f}")

print("\n💾 OUTPUT FILES:")
print("  • Model weights: best_multimodal_model.pth")
print(f"  • Predictions: {submission_file}")

print("\n✨ KEY INNOVATIONS:")
print("  ✓ Structured feature parsing from unstructured text")
print("  ✓ BERT token-level embeddings (not just [CLS])")
print("  ✓ BiLSTM for sequential text aggregation")
print("  ✓ Transformer fusion for cross-modal attention")
print("  ✓ Log-transform target (train on log, evaluate on original)")
print("  ✓ Optuna hyperparameter optimization")
print("  ✓ Unit normalization and derived numeric features")
print("  ✓ Categorical embeddings for unit, brand, weight_category")

print("\n🚀 ADVANTAGES:")
print("  • Captures structured information (value, unit, quantity)")
print("  • Token-level text understanding via BERT")
print("  • Sequential context via BiLSTM")
print("  • Cross-modal fusion via Transformer")
print("  • Robust to missing values and diverse units")
print("  • State-of-the-art multimodal architecture")

print("\n" + "=" * 80)
print("Notebook execution complete! ✨")
print("=" * 80)